In [57]:
import numpy as np
from skimage.io import imread
from skimage.filters import frangi
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize, remove_small_objects
from scipy.ndimage import distance_transform_edt
from Skel2Graph3D import skel2graph3d
from Graph2Skel3D import graph2skel3d
import scipy.io as sio
import os
import time


import sys
sys.path.append("/Users/mutrera/Documents/Uchicago College/Uchicago 2024 - 2025/Summer/Octava/octava/code")
from frangi_2dfilter import frangi_2Dfilter
from fuzzy_thresholding import fuzzy_thresholding
from adaptive_thresholding import adaptive_thresholding

# Functions

In [58]:
# HELPERS 

def BinarizeImage(Im:np.ndarray, frangi_sigma = 8, frangi_enabled = True, method = "Fuzzy Means") -> np.ndarray:
    """
    Binarizes an image using the selected method and options from the GUI app.

    Parameters:
    - app: GUI application object containing UI state (e.g., checkboxes, dropdowns)
    - Im: Input image as a 2D NumPy array (grayscale)

    Returns:
    - Binarized image as a 2D NumPy array (dtype=float)
    """
    # Find the brightest pixel in the image
    col_max_vals = np.max(Im, axis=0) # the maximum of each column
    cc = np.argmax(col_max_vals) #  gets the index of the column where the brightest value occurred
    rr = np.argmax(Im[:, cc]) # captures the row index of that max value

    #	•	rr: row of the brightest pixel
	#   •	cc: column of the brightest pixel
    #   •	So (rr, cc) is the coordinate of the brightest pixel in the whole image.

    # Apply Frangi filter if selected
    if frangi_enabled:
        opts = {
            'sigmarange': [1, frangi_sigma],
            'sigmastepsize': 2,
            'correctionconst1': 0.8,
            'correctionconst2': 15
        }
        Im = frangi_2Dfilter(Im, opts)

    # Apply the selected binarization method
    if method == 'Fuzzy Means':
        nth = 2
        Im = fuzzy_thresholding(Im, nth, 3) - 1

    elif method == 'Adaptive Thresholding':
        Im = adaptive_thresholding(Im, 70).astype(float)

    else:
        raise ValueError("Incorrect Segmentation Method")

    # Ensure vessels are white, background is black, turning the image into black and white
    vessel_value = Im[rr, cc]
    Im[Im == vessel_value] = 255
    Im[Im != 255] = 0

    return Im

def create_binarized_img(paths):
    for path in paths:
        octa = imread(path)
        # Frangi vessel enhancement
        vessels = frangi(
            octa,
            scale_range=(0.5, 4),
            scale_step=0.5,
            alpha=0.5,
            beta=0.5,
            gamma=4,
            black_ridges=False
        )

        adpt_mask = BinarizeImage(
            vessels,
            frangi_sigma=8,
            frangi_enabled=False,
            method="Adaptive Thresholding"
        )

        name = path[-9:]
        name = name.split('.')[0]
        sio.savemat(f"{name}.mat", {f"octa_{name}": adpt_mask.astype(np.bool)})


def skeletonize_main(bin: np.ndarray, THR: int = 0):
    """
    Full skeletonization pipeline with timing instrumentation.
    Returns a dictionary with images, graph objects, and timings.
    """

    total_start = time.time()   # MEASURE TOTAL RUNTIME

    # ------------------------------------------------------
    # Skeletonization
    # ------------------------------------------------------
    y, x = bin.shape
    L = bin.astype(bool)
    sk2 = skeletonize(L)

    # Remove small branches
    sk2 = remove_small_objects(sk2, THR)

    # Local thickness map
    D = distance_transform_edt(L) * 2

    # ------------------------------------------------------
    # Network analysis
    # ------------------------------------------------------
    if np.sum(sk2) == 0:
        nodes = []
        links = []
        return {
            "bin_img": bin,
            "thr": THR,
            "skel_img": sk2,
            "nodes": nodes,
            "links": links,
            "skel2graph_time": 0,
            "graph2skel_time": 0,
            "network_clean_time": 0,
            "total_time": time.time() - total_start
        }

    # 1) First skel2graph call
    t0 = time.time()
    nodes, links = skel2graph3d(sk2, THR)
    t1 = time.time()
    skel2graph_times = [t1 - t0]

    # Compute initial total length
    wl = sum(len(n.links) for n in nodes)

    # 2) First graph2skel call
    t0 = time.time()
    skel2 = graph2skel3d(nodes, links, y, x, 1)
    t1 = time.time()
    graph2skel_times = [t1 - t0]

    # 3) Second skel2graph call
    t0 = time.time()
    nodes, links = skel2graph3d(skel2, 0)
    t1 = time.time()
    skel2graph_times.append(t1 - t0)

    wl_new = sum(len(n.links) for n in nodes)

    # ------------------------------------------------------
    # ITERATIVE NETWORK CLEANING LOOP
    # ------------------------------------------------------
    clean_start = time.time()
    while wl_new != wl:
        wl = wl_new

        # graph2skel
        t0 = time.time()
        skel2 = graph2skel3d(nodes, links, y, x, 1)
        t1 = time.time()
        graph2skel_times.append(t1 - t0)

        # skel2graph
        t0 = time.time()
        nodes, links = skel2graph3d(skel2, 0)
        t1 = time.time()
        skel2graph_times.append(t1 - t0)

        wl_new = sum(len(n.links) for n in nodes)
    clean_end = time.time()

    total_end = time.time()

    # ------------------------------------------------------
    # Return dictionary with all results
    # ------------------------------------------------------
    return {
        "bin_img": bin,
        "thr": THR,
        "skel_img": sk2,
        "nodes": nodes,
        "links": links,
        "skel2graph_time": np.mean(skel2graph_times),
        "graph2skel_time": np.mean(graph2skel_times),
        "network_clean_time": clean_end - clean_start,
        "total_time": total_end - total_start
    }

def iou_skeletons(im1: np.ndarray, im2: np.ndarray) -> float:
    """
    Computes the IoU score between two 2D skeletonized images.
    Both images must be binary (0/1 or bool).

    Returns a float in [0, 1] indicating percentage of pixel overlap.
    """

    # Ensure boolean arrays
    im1 = im1.astype(bool)
    im2 = im2.astype(bool)

    intersection = np.logical_and(im1, im2).sum()
    union = np.logical_or(im1, im2).sum()

    if union == 0:
        return 1.0 if intersection == 0 else 0.0

    return intersection / union

In [59]:
# folder = "/Users/mutrera/Documents/Uchicago College/Uchicago 2024 - 2025/Summer/Octava/octava/assets/MIPs"
# paths = [os.path.join(folder, f) for f in os.listdir(folder)]

# create_binarized_img(paths)

# Tests

In [60]:
folder = "/Users/mutrera/Documents/Uchicago College/Uchicago 2024 - 2025/Summer/Octava/octava/assets/MIPs"
octa_paths = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
]
octa_paths.sort()

folder = "/Users/mutrera/Documents/Uchicago College/Uchicago 2024 - 2025/Summer/Octava/octava/code/Skeletonization/matlab_skels"
matlab_mats = [os.path.join(folder, f) for f in os.listdir(folder)]
matlab_mats.sort()

results = []
for i, path in enumerate(octa_paths):
    octa = imread(path)

    # Frangi vessel enhancement
    vessels = frangi(
        octa,
        scale_range=(0.5, 4),
        scale_step=0.5,
        alpha=0.5,
        beta=0.5,
        gamma=4,
        black_ridges=False
    )

    adpt_mask = BinarizeImage(
        vessels,
        frangi_sigma=8,
        frangi_enabled=False,
        method="Adaptive Thresholding"
    )

    bin_img = adpt_mask
    THR = 1

    result_dict = skeletonize_main(bin_img, THR)
    data = sio.loadmat(matlab_mats[i])

    matlab_skel = data['sk2']
    py_skel = result_dict['skel_img']

    iou = iou_skeletons(matlab_skel, py_skel)
    result_dict['iou'] = iou

    results.append(result_dict)

    print(f"Done with the following image: {octa_paths[i][-9:]} \n")

/var/folders/j8/24ft1_090l56tpp7kwbz30gm0000gn/T/ipykernel_66799/3140863066.py:17: UserWarning: Use keyword parameter `sigmas` instead of `scale_range` and `scale_range` which will be removed in version 0.17.
  vessels = frangi(


Done with the following image: 10301.bmp 



/var/folders/j8/24ft1_090l56tpp7kwbz30gm0000gn/T/ipykernel_66799/3140863066.py:17: UserWarning: Use keyword parameter `sigmas` instead of `scale_range` and `scale_range` which will be removed in version 0.17.
  vessels = frangi(


Done with the following image: 10310.bmp 



/var/folders/j8/24ft1_090l56tpp7kwbz30gm0000gn/T/ipykernel_66799/3140863066.py:17: UserWarning: Use keyword parameter `sigmas` instead of `scale_range` and `scale_range` which will be removed in version 0.17.
  vessels = frangi(


...

Done with the following image: 10346.bmp 



/var/folders/j8/24ft1_090l56tpp7kwbz30gm0000gn/T/ipykernel_66799/3140863066.py:17: UserWarning: Use keyword parameter `sigmas` instead of `scale_range` and `scale_range` which will be removed in version 0.17.
  vessels = frangi(


Done with the following image: 10355.bmp 



/var/folders/j8/24ft1_090l56tpp7kwbz30gm0000gn/T/ipykernel_66799/3140863066.py:17: UserWarning: Use keyword parameter `sigmas` instead of `scale_range` and `scale_range` which will be removed in version 0.17.
  vessels = frangi(


Done with the following image: 10364.bmp 



In [84]:
ious = []
for i, res in enumerate(results):
    img_name = octa_paths[i][-9:]
    iou = res['iou']
    ious.append(iou)

ious = np.array(ious)
print(f"Mean iou score: {round(np.mean(ious) * 100, 3)}%")

Mean iou score: 62.806%


In [85]:
results[0].keys()

dict_keys(['bin_img', 'thr', 'skel_img', 'nodes', 'links', 'skel2graph_time', 'graph2skel_time', 'network_clean_time', 'total_time', 'iou'])

In [86]:
skel2graph_times = []
for i, res in enumerate(results):
    time = res['skel2graph_time']
    skel2graph_times.append(time)

skel2graph_times = np.array(skel2graph_times)
print(f"Mean skel2graph_times: {round(np.mean(skel2graph_times), 3)}s")

Mean skel2graph_times: 6.359s


In [87]:
graph2skel_times = []
for i, res in enumerate(results):
    time = res['graph2skel_time']
    graph2skel_times.append(time)

graph2skel_times = np.array(graph2skel_times)
print(f"Mean graph2skel_times: {round(np.mean(graph2skel_times), 3)}s")

Mean graph2skel_times: 0.005s


In [88]:
network_clean_times = []
for i, res in enumerate(results):
    time = res['network_clean_time']
    network_clean_times.append(time)

network_clean_times = np.array(network_clean_times)
print(f"Mean network_clean_times: {round(np.mean(network_clean_times), 3)}s")

Mean network_clean_times: 31.145s


In [91]:
total_times = []
for i, res in enumerate(results):
    time = res['total_time']
    total_times.append(time)

total_times = np.array(total_times)
print(f"Mean total time: {round(np.mean(total_times), 3)}s")

Mean total time: 45.624s
